# MM/GBSA Free-Energy Analysis

這份 Notebook 一次讀取 1 個 MM/GBSA .dat 檔案。

流程：

1. 指定資料檔、frame 欄位與 energy 欄位。
2. 跳過空行、註解行及無法解析的資料行。
3. 建立時間軸並保存逐 frame free energy。
4. 計算指定 analysis region 的 mean、standard deviation 與資料點數。
5. 輸出 NPZ、CSV、PNG、PDF 與執行摘要。

單一資料檔不代表三個 replicates，因此圖中不畫 replicate mean ± SD error band。


## 1. 環境準備

需要 Python 3、NumPy 與 Matplotlib。

    pip install numpy matplotlib


In [ ]:
# ============================================================
# 2. 載入套件
# ============================================================

from pathlib import Path
import csv

import matplotlib.pyplot as plt
import numpy as np

print(f"NumPy: {np.__version__}")


## 3. 使用者設定區

Python 欄位索引從 0 開始：

- 第 1 欄：index 0
- 第 2 欄：index 1

預設第一欄是 frame index、第二欄是 free energy。若檔案沒有 frame 欄，可將 FRAME_COLUMN_INDEX 設為 None，程式會使用有效資料列的順序建立時間軸。


In [ ]:
# ============================================================
# 4. 使用者設定區：一般情況只修改這一區
# ============================================================

DATA_FILE = Path(
    "/ceph/sharedfs/work/MYTLab/asher/center_ion_project/na/"
    "mmgbsa_control_12_15/mmgbsa606_rep1/"
    "final_results_rep1.dat"
)

SYSTEM_NAME = "Na_606_rep1"
ION_LABEL = "Na+"
DOCKING_SCORE = "6.06"
CURVE_COLOR = "royalblue"

# Python 使用零起始欄位索引。
FRAME_COLUMN_INDEX = 0
ENERGY_COLUMN_INDEX = 1

# 第一欄每增加 1 frame，代表 0.002 ns。
TIME_PER_FRAME_NS = 0.002

# 從此時間開始計算 analysis-region 統計。
ANALYSIS_START_NS = 50.0

# 圖片 Y 軸範圍；設為 None 時自動決定。
Y_LIMIT_KCAL_MOL = (-25.0, 1.0)

OUTPUT_DIR = Path("./results") / SYSTEM_NAME

DISPLAY_LABEL = (
    f"{ION_LABEL} — Docking score = {DOCKING_SCORE}"
)

print(f"System: {SYSTEM_NAME}")
print(f"Input file: {DATA_FILE}")
print(f"Energy column index: {ENERGY_COLUMN_INDEX}")
print(f"Analysis region starts at: {ANALYSIS_START_NS} ns")
print(f"Output directory: {OUTPUT_DIR.resolve()}")


## 5. 資料讀取函數

讀取規則：

- 跳過空行
- 跳過以 # 或 @ 開頭的行
- energy 必須能轉換為有限浮點數
- 若指定 frame 欄位，frame 必須為有限數值且嚴格遞增

程式會回報略過多少行，避免資料格式錯誤卻沒有被注意。


In [ ]:
# ============================================================
# 6. 讀取與檢查函數
# ============================================================

COMMENT_PREFIXES = ("#", "@")


def parse_energy_file(
    file_path,
    energy_column_index,
    frame_column_index=None,
):
    """
    讀取單一 MM/GBSA 數據檔。

    Parameters
    ----------
    file_path : pathlib.Path
        輸入資料檔。
    energy_column_index : int
        Free-energy 欄位的零起始索引。
    frame_column_index : int or None
        Frame 欄位的零起始索引；None 表示使用有效資料列順序。

    Returns
    -------
    frame_values : numpy.ndarray
        原始 frame 欄位，或從 0 開始的有效資料列索引。
    energy_kcal_mol : numpy.ndarray
        Free energy，單位為 kcal/mol。
    parse_summary : dict
        總行數、註解／空行數與無效行數。
    """
    if energy_column_index < 0:
        raise ValueError("ENERGY_COLUMN_INDEX 不可小於 0。")

    if frame_column_index is not None and frame_column_index < 0:
        raise ValueError("FRAME_COLUMN_INDEX 不可小於 0。")

    required_column = max(
        energy_column_index,
        (
            frame_column_index
            if frame_column_index is not None
            else energy_column_index
        ),
    )

    frame_values = []
    energy_values = []
    total_lines = 0
    skipped_comment_or_blank = 0
    skipped_invalid = 0

    with file_path.open("r", encoding="utf-8") as input_file:
        for line_number, raw_line in enumerate(input_file, start=1):
            total_lines += 1
            line = raw_line.strip()

            if not line or line.startswith(COMMENT_PREFIXES):
                skipped_comment_or_blank += 1
                continue

            parts = line.split()

            if len(parts) <= required_column:
                skipped_invalid += 1
                continue

            try:
                energy_value = float(parts[energy_column_index])

                if frame_column_index is None:
                    frame_value = float(len(energy_values))
                else:
                    frame_value = float(parts[frame_column_index])

            except ValueError:
                skipped_invalid += 1
                continue

            if not (
                np.isfinite(energy_value)
                and np.isfinite(frame_value)
            ):
                skipped_invalid += 1
                continue

            frame_values.append(frame_value)
            energy_values.append(energy_value)

    if not energy_values:
        raise ValueError(
            "沒有讀到有效能量資料。"
            "請檢查 ENERGY_COLUMN_INDEX 與資料檔格式。"
        )

    if len(energy_values) < 2:
        raise ValueError(
            "至少需要兩筆有效資料，才能建立時間序列。"
        )

    frame_values = np.asarray(frame_values, dtype=float)
    energy_values = np.asarray(energy_values, dtype=float)

    if np.any(np.diff(frame_values) <= 0):
        raise ValueError(
            "Frame values 必須嚴格遞增。"
            "請檢查 FRAME_COLUMN_INDEX 或資料排序。"
        )

    parse_summary = {
        "total_lines": total_lines,
        "valid_rows": len(energy_values),
        "skipped_comment_or_blank": skipped_comment_or_blank,
        "skipped_invalid": skipped_invalid,
    }

    return frame_values, energy_values, parse_summary


## 7. 執行讀取、時間換算與統計

時間以第一個有效 frame 為 0 ns：

    time_ns = (frame_value - first_frame_value) × TIME_PER_FRAME_NS

如果第一欄其實是 NAMD step 而不是 DCD frame index，請修改 TIME_PER_FRAME_NS，使其符合該欄位每增加 1 所代表的時間。


In [ ]:
# ============================================================
# 8. 執行分析並儲存 NPZ、CSV 與摘要
# ============================================================

if not DATA_FILE.is_file():
    raise FileNotFoundError(
        f"找不到資料檔：{DATA_FILE}\n"
        "請回到使用者設定區修改 DATA_FILE。"
    )

if TIME_PER_FRAME_NS <= 0:
    raise ValueError("TIME_PER_FRAME_NS 必須大於 0。")

frame_values, energy_kcal_mol, parse_summary = parse_energy_file(
    DATA_FILE,
    ENERGY_COLUMN_INDEX,
    FRAME_COLUMN_INDEX,
)

time_ns = (
    frame_values - frame_values[0]
) * TIME_PER_FRAME_NS

analysis_mask = time_ns >= ANALYSIS_START_NS
analysis_count = int(np.count_nonzero(analysis_mask))

if analysis_count > 0:
    analysis_mean = float(
        np.mean(energy_kcal_mol[analysis_mask])
    )
    analysis_std = float(
        np.std(energy_kcal_mol[analysis_mask])
    )
else:
    analysis_mean = np.nan
    analysis_std = np.nan
    print(
        "[Warning] 軌跡時間未到 ANALYSIS_START_NS，"
        "analysis mean 與 SD 將保存為 NaN。"
    )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

npz_path = OUTPUT_DIR / "mmgbsa_energy_data.npz"
np.savez_compressed(
    npz_path,
    system_name=np.asarray(SYSTEM_NAME),
    display_label=np.asarray(DISPLAY_LABEL),
    curve_color=np.asarray(CURVE_COLOR),
    frame_values=frame_values,
    time_ns=time_ns,
    energy_kcal_mol=energy_kcal_mol,
    frame_column_index=np.asarray(
        -1 if FRAME_COLUMN_INDEX is None else FRAME_COLUMN_INDEX
    ),
    energy_column_index=np.asarray(ENERGY_COLUMN_INDEX),
    time_per_frame_ns=np.asarray(TIME_PER_FRAME_NS),
    analysis_start_ns=np.asarray(ANALYSIS_START_NS),
    analysis_mean_kcal_mol=np.asarray(analysis_mean),
    analysis_std_kcal_mol=np.asarray(analysis_std),
    analysis_count=np.asarray(analysis_count),
)

csv_path = OUTPUT_DIR / "mmgbsa_energy_data.csv"
with csv_path.open("w", newline="", encoding="utf-8") as csv_file:
    writer = csv.writer(csv_file)
    writer.writerow([
        "data_index",
        "frame_value",
        "time_ns",
        "free_energy_kcal_mol",
        "in_analysis_region",
    ])

    for data_index, values in enumerate(
        zip(
            frame_values,
            time_ns,
            energy_kcal_mol,
            analysis_mask,
        )
    ):
        writer.writerow([
            data_index,
            f"{values[0]:.6f}",
            f"{values[1]:.6f}",
            f"{values[2]:.6f}",
            bool(values[3]),
        ])

summary_path = OUTPUT_DIR / "run_summary.txt"
with summary_path.open("w", encoding="utf-8") as summary_file:
    summary_file.write(f"System: {SYSTEM_NAME}\n")
    summary_file.write(f"Display label: {DISPLAY_LABEL}\n")
    summary_file.write(f"Input file: {DATA_FILE}\n")
    summary_file.write(
        f"Frame column index: {FRAME_COLUMN_INDEX}\n"
    )
    summary_file.write(
        f"Energy column index: {ENERGY_COLUMN_INDEX}\n"
    )
    summary_file.write(
        f"Time per frame value: {TIME_PER_FRAME_NS} ns\n"
    )
    summary_file.write(
        f"Total lines: {parse_summary['total_lines']}\n"
    )
    summary_file.write(
        f"Valid rows: {parse_summary['valid_rows']}\n"
    )
    summary_file.write(
        "Skipped comment/blank lines: "
        f"{parse_summary['skipped_comment_or_blank']}\n"
    )
    summary_file.write(
        f"Skipped invalid lines: {parse_summary['skipped_invalid']}\n"
    )
    summary_file.write(
        f"Final time: {time_ns[-1]:.6f} ns\n"
    )
    summary_file.write(
        f"Analysis start: {ANALYSIS_START_NS} ns\n"
    )
    summary_file.write(
        f"Analysis data points: {analysis_count}\n"
    )
    summary_file.write(
        f"Analysis mean: {analysis_mean:.6f} kcal/mol\n"
    )
    summary_file.write(
        f"Analysis SD: {analysis_std:.6f} kcal/mol\n"
    )

print(f"Valid rows: {parse_summary['valid_rows']:,}")
print(f"Skipped invalid rows: {parse_summary['skipped_invalid']:,}")
print(f"Final time: {time_ns[-1]:.3f} ns")
print(
    f"Analysis region: {ANALYSIS_START_NS:.3f}–"
    f"{time_ns[-1]:.3f} ns"
)
print(
    f"Analysis mean ± SD: "
    f"{analysis_mean:.3f} ± {analysis_std:.3f} kcal/mol"
)
print(f"NPZ saved: {npz_path.resolve()}")
print(f"CSV saved: {csv_path.resolve()}")
print(f"Summary saved: {summary_path.resolve()}")


## 9. 從 NPZ 載入並繪圖

若只需修改圖形，可從這裡往下執行，不必重新解析原始 .dat。


In [ ]:
# ============================================================
# 10. 載入 NPZ
# ============================================================

npz_path = OUTPUT_DIR / "mmgbsa_energy_data.npz"

if not npz_path.is_file():
    raise FileNotFoundError(
        f"找不到 NPZ：{npz_path}\n"
        "請先執行 MM/GBSA 分析 cell。"
    )

data = np.load(npz_path, allow_pickle=False)

required_keys = {
    "display_label",
    "curve_color",
    "time_ns",
    "energy_kcal_mol",
    "analysis_start_ns",
}

missing_keys = required_keys.difference(data.files)

if missing_keys:
    raise KeyError(f"NPZ 缺少欄位：{sorted(missing_keys)}")

plot_label = str(data["display_label"])
plot_color = str(data["curve_color"])
plot_time_ns = data["time_ns"]
plot_energy = data["energy_kcal_mol"]
plot_analysis_start = float(data["analysis_start_ns"])

if len(plot_time_ns) != len(plot_energy):
    raise ValueError("NPZ 中 time_ns 與 energy 長度不同。")

print(f"Loaded: {npz_path.resolve()}")
print(f"Data points: {len(plot_time_ns):,}")


## 11. 繪製 free-energy time series

灰色區域代表 analysis region。單一資料檔只畫逐 frame energy，不畫 replicate error band。


In [ ]:
# ============================================================
# 12. MM/GBSA Free-Energy 圖
# ============================================================

fig, ax = plt.subplots(figsize=(13, 10), dpi=300)

ax.plot(
    plot_time_ns,
    plot_energy,
    color=plot_color,
    linewidth=2.5,
    linestyle="-",
    alpha=0.9,
    label=plot_label,
)

plot_end_ns = float(plot_time_ns[-1])

if plot_analysis_start <= plot_end_ns:
    ax.axvspan(
        plot_analysis_start,
        plot_end_ns,
        color="grey",
        alpha=0.10,
        linewidth=0,
        label="_nolegend_",
    )

ax.set_title(
    "MM/GBSA Free-Energy Time Series",
    fontsize=30,
    pad=18,
)
ax.set_xlabel(
    "Time (ns)",
    fontsize=30,
    labelpad=12,
)
ax.set_ylabel(
    "Free Energy (kcal/mol)",
    fontsize=30,
    labelpad=12,
)

ax.set_xlim(
    float(plot_time_ns[0]),
    plot_end_ns,
)

if Y_LIMIT_KCAL_MOL is not None:
    ax.set_ylim(*Y_LIMIT_KCAL_MOL)

ax.tick_params(
    axis="both",
    which="major",
    labelsize=24,
)

ax.grid(
    axis="y",
    linestyle="--",
    alpha=0.4,
)

ax.legend(
    fontsize=18,
    loc="lower left",
    frameon=True,
    framealpha=0.9,
)

fig.tight_layout()

png_path = OUTPUT_DIR / "figure_mmgbsa_free_energy.png"
pdf_path = OUTPUT_DIR / "figure_mmgbsa_free_energy.pdf"

fig.savefig(
    png_path,
    dpi=300,
    bbox_inches="tight",
)
fig.savefig(
    pdf_path,
    bbox_inches="tight",
)

plt.show()

print(f"PNG saved: {png_path.resolve()}")
print(f"PDF saved: {pdf_path.resolve()}")


## 13. 完成後確認

- ENERGY_COLUMN_INDEX 指向真正的 MM/GBSA free-energy 欄位。
- FRAME_COLUMN_INDEX 指向 frame index；若檔案沒有 frame 欄，設為 None。
- 第一個有效資料點的時間為 0 ns。
- TIME_PER_FRAME_NS 與資料檔第一欄的定義一致。
- Analysis region 有資料點，mean 與 SD 不是 NaN。
- 單一資料檔圖沒有 replicate mean、error band 或標記點。
- 若圖形超出固定 Y 軸，將 Y_LIMIT_KCAL_MOL 設為 None。

分析下一個檔案時，只需更換 DATA_FILE、SYSTEM_NAME、DOCKING_SCORE、CURVE_COLOR，再由上往下執行。
